In [ ]:
!pip install pyspark pandas pyarrow -q

from google.colab import drive
drive.mount("/content/drive")

import os
import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder
    .appName("HM_Feature_Engineering")
    .config("spark.driver.memory", "8g")
    .config("spark.memory.offHeap.enabled", "true")
    .config("spark.memory.offHeap.size", "2g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

print("Spark da khoi tao xong.")

Mounted at /content/drive
Spark da khoi tao xong.


In [ ]:
BASE_PATH = "/content/drive/MyDrive/HM-DATA/"

INPUT_TRANS = BASE_PATH + "processed_v2/cleaned_transactions.parquet"
INPUT_CUST = BASE_PATH + "processed_v2/customers_processed.parquet"
INPUT_ART = BASE_PATH + "processed_v2/articles_processed.parquet"

MASTER_CAND_DIR = BASE_PATH + "outputs_v2/master/"
LABELED_DIR = BASE_PATH + "outputs_v2/labeled/"
FEATURE_DIR = BASE_PATH + "outputs_v2/features/"

os.makedirs(FEATURE_DIR, exist_ok=True)

TRAIN_LABELED_PATH = LABELED_DIR + "train_labeled_sampled_10_1.parquet"
TEST_BASE_PATH = LABELED_DIR + "test_base_candidates.parquet"

TRAIN_FEATURE_PATH = FEATURE_DIR + "train_features.parquet"
TEST_FEATURE_PATH = FEATURE_DIR + "test_features.parquet"

print("Train labeled:", TRAIN_LABELED_PATH)
print("Test base    :", TEST_BASE_PATH)
print("Train output :", TRAIN_FEATURE_PATH)
print("Test output  :", TEST_FEATURE_PATH)

Train labeled: /content/drive/MyDrive/HM-DATA/outputs_v2/labeled/train_labeled_sampled_10_1.parquet
Test base    : /content/drive/MyDrive/HM-DATA/outputs_v2/labeled/test_base_candidates.parquet
Train output : /content/drive/MyDrive/HM-DATA/outputs_v2/features/train_features.parquet
Test output  : /content/drive/MyDrive/HM-DATA/outputs_v2/features/test_features.parquet


In [ ]:
transactions = spark.read.parquet(INPUT_TRANS)
customers = spark.read.parquet(INPUT_CUST)
articles = spark.read.parquet(INPUT_ART)

x_train_base = spark.read.parquet(TRAIN_LABELED_PATH)
x_test_base = spark.read.parquet(TEST_BASE_PATH)

print("Train labeled:", x_train_base.count())
print("Test base    :", x_test_base.count())
print("Transactions :", transactions.count())
print("Customers    :", customers.count())
print("Articles     :", articles.count())

Train labeled: 230601
Test base    : 53853851
Transactions : 31788324
Customers    : 1371980
Articles     : 105542


In [ ]:
max_date = transactions.select(F.max("t_dat_date")).collect()[0][0]

test_start = max_date - datetime.timedelta(days=7)
val_start = test_start - datetime.timedelta(days=7)

print("Max date  :", max_date)
print("Val start :", val_start)
print("Test start:", test_start)

Max date  : 2020-09-22
Val start : 2020-09-08
Test start: 2020-09-15


In [ ]:
def prepare_base_candidates(base_df):
    df = base_df

    if "als_score" not in df.columns:
        df = df.withColumn("als_score", F.lit(0.0))

    if "itemcf_score" not in df.columns:
        df = df.withColumn("itemcf_score", F.lit(0.0))

    if "sources" not in df.columns:
        df = df.withColumn("sources", F.array().cast("array<string>"))

    return df

In [ ]:
def calculate_features_for_window(base_df, end_date, window_days=42):
    start_date = end_date - datetime.timedelta(days=window_days)
    base_df = prepare_base_candidates(base_df)

    article_meta = articles.select(
        "article_id",
        F.col("product_type_name").alias("item_product_type"),
        F.col("colour_group_name").alias("item_colour_group")
    )

    customer_meta = (
        customers
        .select("customer_id", "age")
        .withColumn(
            "user_age_group",
            F.when(F.col("age") < 25, "<25")
             .when((F.col("age") >= 25) & (F.col("age") <= 35), "25-35")
             .when((F.col("age") >= 36) & (F.col("age") <= 45), "36-45")
             .when((F.col("age") >= 46) & (F.col("age") <= 55), "46-55")
             .otherwise(">55")
        )
    )

    hist_trans = (
        transactions
        .filter((F.col("t_dat_date") >= start_date) & (F.col("t_dat_date") < end_date))
    )

    price_median_list = hist_trans.approxQuantile("price", [0.5], 0.01)
    global_median_price = price_median_list[0] if price_median_list and price_median_list[0] is not None else 0.02

    age_median_list = customers.select("age").na.drop().approxQuantile("age", [0.5], 0.01)
    global_median_age = age_median_list[0] if age_median_list and age_median_list[0] is not None else 25

    hist_enriched = hist_trans.join(
        F.broadcast(article_meta),
        "article_id",
        "inner"
    )

    item_features = (
        hist_enriched
        .groupBy("article_id")
        .agg(
            F.count("customer_id").alias("item_total_sales"),
            F.avg("price").alias("item_avg_price"),
            F.max("price").alias("item_max_price"),
            ((F.count("customer_id") - F.countDistinct("customer_id")) / F.count("customer_id")).alias("item_repurchase_ratio"),
            F.avg(F.when(F.col("sales_channel_id") == 2, 1.0).otherwise(0.0)).alias("item_online_ratio"),
            F.datediff(F.lit(end_date), F.min("t_dat_date")).alias("item_days_since_first_sale"),
            F.datediff(F.lit(end_date), F.max("t_dat_date")).alias("item_days_since_last_sale"),
            (F.datediff(F.max("t_dat_date"), F.min("t_dat_date")) + 1).alias("item_active_days")
        )
        .withColumn("item_price_drop_ratio", F.col("item_avg_price") / F.col("item_max_price"))
        .drop("item_max_price")
    )

    user_features = (
        hist_enriched
        .groupBy("customer_id")
        .agg(
            F.count("article_id").alias("user_total_purchases"),
            F.avg("price").alias("user_avg_price"),
            F.stddev("price").alias("user_price_std"),
            F.countDistinct("item_product_type").alias("user_unique_product_type_count"),
            F.avg(F.when(F.col("sales_channel_id") == 2, 1.0).otherwise(0.0)).alias("user_online_ratio"),
            F.datediff(F.lit(end_date), F.max("t_dat_date")).alias("user_days_since_last_purchase"),
            F.sum(F.when(F.col("t_dat_date") >= F.date_sub(F.lit(end_date), 7), 1).otherwise(0)).alias("user_purchase_count_7d"),
            F.sum(F.when(F.col("t_dat_date") >= F.date_sub(F.lit(end_date), 14), 1).otherwise(0)).alias("user_purchase_count_14d")
        )
    )

    user_item_features = (
        hist_trans
        .groupBy("customer_id", "article_id")
        .agg(
            F.count("t_dat_date").alias("user_item_buy_count"),
            F.datediff(F.lit(end_date), F.max("t_dat_date")).alias("user_item_days_since_last_buy")
        )
    )

    user_product_type_features = (
        hist_enriched
        .groupBy("customer_id", "item_product_type")
        .agg(F.count("*").alias("user_product_type_buy_count"))
    )

    user_colour_features = (
        hist_enriched
        .groupBy("customer_id", "item_colour_group")
        .agg(F.count("*").alias("user_colour_buy_count"))
    )

    item_trend_features = (
        hist_trans
        .groupBy("article_id")
        .agg(
            F.sum(F.when(F.col("t_dat_date") >= F.date_sub(F.lit(end_date), 3), 1).otherwise(0)).alias("item_sales_3d"),
            F.sum(F.when(F.col("t_dat_date") >= F.date_sub(F.lit(end_date), 7), 1).otherwise(0)).alias("item_sales_7d"),
            F.sum(F.when(F.col("t_dat_date") >= F.date_sub(F.lit(end_date), 14), 1).otherwise(0)).alias("item_sales_14d")
        )
    )

    hist_with_age = hist_trans.join(
        F.broadcast(customer_meta),
        "customer_id",
        "inner"
    )

    item_age_features = (
        hist_with_age
        .groupBy("article_id", "user_age_group")
        .agg(
            F.count("*").alias("item_age_group_total_sales"),
            F.sum(F.when(F.col("t_dat_date") >= F.date_sub(F.lit(end_date), 7), 1).otherwise(0)).alias("item_age_group_sales_7d")
        )
    )

    item_avg_age_features = (
        hist_with_age
        .groupBy("article_id")
        .agg(F.avg("age").alias("item_avg_customer_age"))
    )

    df = (
        base_df
        .join(F.broadcast(customer_meta), "customer_id", "left")
        .join(F.broadcast(article_meta), "article_id", "left")
        .join(F.broadcast(item_features), "article_id", "left")
        .join(F.broadcast(user_features), "customer_id", "left")
        .join(user_item_features, ["customer_id", "article_id"], "left")
        .join(F.broadcast(item_trend_features), "article_id", "left")
        .join(F.broadcast(item_avg_age_features), "article_id", "left")
        .join(F.broadcast(user_product_type_features), ["customer_id", "item_product_type"], "left")
        .join(F.broadcast(user_colour_features), ["customer_id", "item_colour_group"], "left")
        .join(F.broadcast(item_age_features), ["article_id", "user_age_group"], "left")
    )

    fill_values = {
        "age": global_median_age,
        "item_product_type": "Unknown",
        "item_colour_group": "Unknown",

        "item_total_sales": 0,
        "item_avg_price": global_median_price,
        "item_price_drop_ratio": 1.0,
        "item_repurchase_ratio": 0.0,
        "item_online_ratio": 0.5,
        "item_days_since_first_sale": 999,
        "item_days_since_last_sale": 999,
        "item_active_days": 0,

        "user_total_purchases": 0,
        "user_avg_price": global_median_price,
        "user_price_std": 0.0,
        "user_unique_product_type_count": 0,
        "user_online_ratio": 0.5,
        "user_days_since_last_purchase": 999,
        "user_purchase_count_7d": 0,
        "user_purchase_count_14d": 0,

        "user_item_buy_count": 0,
        "user_item_days_since_last_buy": 999,

        "user_product_type_buy_count": 0,
        "user_colour_buy_count": 0,

        "item_sales_3d": 0,
        "item_sales_7d": 0,
        "item_sales_14d": 0,

        "item_avg_customer_age": global_median_age,
        "item_age_group_total_sales": 0,
        "item_age_group_sales_7d": 0,

        "als_score": 0.0,
        "itemcf_score": 0.0
    }

    df = df.fillna(fill_values)

    df = (
        df
        .withColumn("user_item_price_diff", F.abs(F.col("item_avg_price") - F.col("user_avg_price")))
        .withColumn("user_item_age_diff", F.abs(F.col("age") - F.col("item_avg_customer_age")))
        .withColumn("item_trend_velocity_7d_14d", F.col("item_sales_7d") / (F.col("item_sales_14d") + 1.0))
        .withColumn("user_item_channel_diff", F.abs(F.col("user_online_ratio") - F.col("item_online_ratio")))
        .withColumn("user_product_type_ratio", F.col("user_product_type_buy_count") / (F.col("user_total_purchases") + 1.0))
        .withColumn("user_colour_ratio", F.col("user_colour_buy_count") / (F.col("user_total_purchases") + 1.0))
        .withColumn("item_age_group_trend_ratio_7d", F.col("item_age_group_sales_7d") / (F.col("item_sales_7d") + 1.0))
        .withColumn("source_from_als", F.when(F.array_contains(F.col("sources"), "als"), 1).otherwise(0))
        .withColumn("source_from_itemcf", F.when(F.array_contains(F.col("sources"), "itemcf"), 1).otherwise(0))
        .withColumn("source_count", F.coalesce(F.size(F.col("sources")), F.lit(0)))
    )

    df = df.drop("sources", "user_age_group")

    return df

In [ ]:
print("Dang tinh feature cho train...")

train_features = calculate_features_for_window(
    base_df=x_train_base,
    end_date=val_start,
    window_days=42
)

print("Dang tinh feature cho test...")

test_features = calculate_features_for_window(
    base_df=x_test_base,
    end_date=test_start,
    window_days=42
)

print("Train columns:", len(train_features.columns))
print("Test columns :", len(test_features.columns))

Dang tinh feature cho train...
Dang tinh feature cho test...
Train columns: 44
Test columns : 43


In [ ]:
train_features.write.mode("overwrite").parquet(TRAIN_FEATURE_PATH)
test_features.write.mode("overwrite").parquet(TEST_FEATURE_PATH)

print("Da luu train feature:", TRAIN_FEATURE_PATH)
print("Da luu test feature :", TEST_FEATURE_PATH)

Da luu train feature: /content/drive/MyDrive/HM-DATA/outputs_v2/features/train_features.parquet
Da luu test feature : /content/drive/MyDrive/HM-DATA/outputs_v2/features/test_features.parquet


In [ ]:
id_cols = ["customer_id", "article_id", "label"]
feature_cols = [c for c in train_features.columns if c not in id_cols]

print("So feature:", len(feature_cols))

for col_name in feature_cols:
    print(col_name)

So feature: 41
item_colour_group
item_product_type
als_score
itemcf_score
source_count
age
item_total_sales
item_avg_price
item_repurchase_ratio
item_online_ratio
item_days_since_first_sale
item_days_since_last_sale
item_active_days
item_price_drop_ratio
user_total_purchases
user_avg_price
user_price_std
user_unique_product_type_count
user_online_ratio
user_days_since_last_purchase
user_purchase_count_7d
user_purchase_count_14d
user_item_buy_count
user_item_days_since_last_buy
item_sales_3d
item_sales_7d
item_sales_14d
item_avg_customer_age
user_product_type_buy_count
user_colour_buy_count
item_age_group_total_sales
item_age_group_sales_7d
user_item_price_diff
user_item_age_diff
item_trend_velocity_7d_14d
user_item_channel_diff
user_product_type_ratio
user_colour_ratio
item_age_group_trend_ratio_7d
source_from_als
source_from_itemcf
